# PDEBench-Lang: BART Baseline for PDE Classification

**Goal:** First-pass notebook baseline using T5-small with natural language dialect input.

**Target:** Single concatenated seq2seq output containing:
- Behavioral label (PDE family)
- Operators
- Reasoning chain

**Budget:** 3 epochs on full dataset

## Phase 1: Data/Task Setup

In [1]:
# Install required packages in the active kernel environment
import sys
!{sys.executable} -m pip install -q "transformers[torch]" "accelerate>=1.1.0" datasets evaluate rouge_score scikit-learn

In [2]:
import json
import os
import random
import numpy as np
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    BartTokenizer,
    BartForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
import evaluate

# Set reproducible config
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Configuration
CONFIG = {
    "model_name": "facebook/bart-base",
    "max_input_length": 128,
    "max_output_length": 256,
    "learning_rate": 3e-4,
    "batch_size": 16,
    "num_epochs": 3,
    "val_split": 0.1,
    "seed": SEED,
}
DIALECTS = ["natural", "latex", "prefix", "postfix"]
MODELS_DIR = "./models"
LOG_DIR = "./logs"
os.environ["TENSORBOARD_LOGGING_DIR"] = LOG_DIR
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Configuration: {CONFIG}")
print(f"Dialects: {DIALECTS}")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # Mac with Apple Silicon
    device = torch.device("mps")
else:
    device = torch.device("cpu")

use_cuda = device.type == "cuda"
if use_cuda:

    torch.backends.cudnn.benchmark = True

print(f"Using device: {device}")
print(f"CUDA acceleration enabled: {use_cuda}")

Configuration: {'model_name': 'facebook/bart-base', 'max_input_length': 128, 'max_output_length': 256, 'learning_rate': 0.0003, 'batch_size': 16, 'num_epochs': 3, 'val_split': 0.1, 'seed': 42}
Dialects: ['natural', 'latex', 'prefix', 'postfix']
Using device: mps
CUDA acceleration enabled: False


In [3]:
# Load dataset and validate schema
def load_dataset_jsonl(filepath):
    """Load dataset from JSONL file and validate required fields."""
    data = []
    required_fields = ["family", "dialects", "labels"]

    with open(filepath, "r") as f:
        for line_num, line in enumerate(f, 1):
            instance = json.loads(line.strip())

            # Validate schema
            for field in required_fields:
                if field not in instance:
                    raise ValueError(f"Line {line_num}: Missing required field '{field}'")

            # Validate nested fields
            for dialect in DIALECTS:
                if dialect not in instance["dialects"]:
                    raise ValueError(f"Line {line_num}: Missing 'dialects.{dialect}'")
            if "behavioral" not in instance["labels"]:
                raise ValueError(f"Line {line_num}: Missing 'labels.behavioral'")
            if "operators" not in instance["labels"]:
                raise ValueError(f"Line {line_num}: Missing 'labels.operators'")
            if "reasoning" not in instance["labels"]:
                raise ValueError(f"Line {line_num}: Missing 'labels.reasoning'")

            data.append(instance)

    return data

# Load the dataset
dataset_path = "varied_data_generation/dataset.jsonl"
raw_data = load_dataset_jsonl(dataset_path)

print(f"Loaded {len(raw_data)} instances")
print(f"\nSample instance:")
print(json.dumps(raw_data[0], indent=2))

Loaded 10000 instances

Sample instance:
{
  "family": "Advection",
  "coefficients": {
    "c": 1.23
  },
  "dialects": {
    "latex": "\\frac{\\partial}{\\partial t} u{\\left(t,x \\right)} + 1.23 \\frac{\\partial}{\\partial x} u{\\left(t,x \\right)} = 0",
    "prefix": "=(+(*(1.23000000000000, d(u(t, x), x)), d(u(t, x), t)), 0)",
    "postfix": "1.23000000000000 u(t, x) x d * u(t, x) t d + 0 =",
    "natural": "u propagates without diffusion: u_t plus 1.23 times u_x equals zero."
  },
  "labels": {
    "behavioral": "Advection",
    "operators": [
      "exp",
      "polynomial"
    ],
    "reasoning": "No second-order spatial derivative; only u_t and 1.23*u_x appear \u2014 this is advection, not diffusion, at speed 1.23."
  }
}


In [4]:
# Analyze family distribution
from collections import Counter

family_counts = Counter(instance["family"] for instance in raw_data)
print("PDE Family Distribution:")
print("-" * 30)
for family, count in sorted(family_counts.items()):
    print(f"{family}: {count} ({count/len(raw_data)*100:.1f}%)")

PDE Family Distribution:
------------------------------
Advection: 2000 (20.0%)
Burgers: 2000 (20.0%)
Heat: 2000 (20.0%)
Laplace: 2000 (20.0%)
Wave: 2000 (20.0%)


In [5]:
# Define input extractor for BART classification
def extract_text(instance, dialect="natural"):
    """Return a PDE description in the selected dialect."""
    return instance["dialects"][dialect]

# Extract the label (PDE family)
def extract_label(instance):
    """Return the PDE family label."""
    return instance["labels"]["behavioral"]

# Preview example
sample = raw_data[10]

print("Input samples by dialect:")
for dialect in DIALECTS:
    print(f"{dialect}: {extract_text(sample, dialect)}")

print("\nLabel:")
print(extract_label(sample))

Input samples by dialect:
natural: Wave propagation with speed 2.0: the second-order time change equals 2.0^2 times the second-order spatial change.
latex: \frac{\partial^{2}}{\partial t^{2}} u{\left(t,x \right)} = 4.0 \frac{\partial^{2}}{\partial x^{2}} u{\left(t,x \right)}
prefix: =(d(d(u(t, x), t), t), *(4.00000000000000, d(d(u(t, x), x), x)))
postfix: u(t, x) t d t d 4.00000000000000 u(t, x) x d x d * =

Label:
Wave


In [6]:
# Build label mappings
families = sorted(set(extract_label(d) for d in raw_data))

label2id = {f: i for i, f in enumerate(families)}
id2label = {i: f for f, i in label2id.items()}

print("Label mapping:")
print(label2id)

Label mapping:
{'Advection': 0, 'Burgers': 1, 'Heat': 2, 'Laplace': 3, 'Wave': 4}


## Phase 2: Training Pipeline

In [7]:
# Build stratified train/validation/test split by family
labels = [instance["family"] for instance in raw_data]

train_val, test_data = train_test_split(
    raw_data,
    test_size=0.1,
    stratify=labels,
    random_state=CONFIG["seed"]
)

train_data, val_data = train_test_split(
    train_val,
    test_size=CONFIG["val_split"] / (1.0 - CONFIG["val_split"]),
    stratify=[d["family"] for d in train_val],
    random_state=CONFIG["seed"],
)

print(f"Train size: {len(train_data)}")
print(f"Validation size: {len(val_data)}")
print(f"Test size: {len(test_data)}")

# Verify balanced distribution
train_families = Counter(d["family"] for d in train_data)
val_families = Counter(d["family"] for d in val_data)
test_families = Counter(d["family"] for d in test_data)

print("\nTrain distribution:")
for family, count in sorted(train_families.items()):
    print(f"  {family}: {count}")

print("\nValidation distribution:")
for family, count in sorted(val_families.items()):
    print(f"  {family}: {count}")

print("\nTest distribution:")
for family, count in sorted(test_families.items()):
    print(f"  {family}: {count}")

Train size: 7999
Validation size: 1001
Test size: 1000

Train distribution:
  Advection: 1600
  Burgers: 1600
  Heat: 1600
  Laplace: 1599
  Wave: 1600

Validation distribution:
  Advection: 200
  Burgers: 200
  Heat: 200
  Laplace: 201
  Wave: 200

Test distribution:
  Advection: 200
  Burgers: 200
  Heat: 200
  Laplace: 200
  Wave: 200


In [8]:
# Initialize tokenizer and model
# Note: `facebook/bart-base` does not include a pretrained classification head,
# so the classification head weights are initialized randomly and must be trained.
tokenizer = BartTokenizer.from_pretrained(CONFIG["model_name"])
model = BartForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)
model.to(device)


print(f"Model loaded: {CONFIG['model_name']}")
print(f"Model parameters: {model.num_parameters():,}")

Some weights of BartForSequenceClassification were not initialized from the model checkpoint at facebook/bart-base and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: facebook/bart-base
Model parameters: 140,014,853


In [9]:
# Preprocess and tokenize for BART classification

def preprocess_data(data):
    """Convert raw data into text and label."""

    texts = [d["dialects"]["natural"] for d in data]
    labels = [label2id[d["labels"]["behavioral"]] for d in data]

    return {
        "text": texts,
        "label": labels
    }


def tokenize_function(examples):
    """Tokenize text inputs for BART."""

    tokenized = tokenizer(
        examples["text"],
        max_length=CONFIG["max_input_length"],
        truncation=True,
        padding="max_length"
    )

    tokenized["labels"] = examples["label"]

    return tokenized


# Create datasets
train_processed = preprocess_data(train_data)
val_processed = preprocess_data(val_data)

train_dataset = Dataset.from_dict(train_processed)
val_dataset = Dataset.from_dict(val_processed)


# Tokenize datasets
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text", "label"]
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text", "label"]
)


print(f"Train dataset: {len(train_tokenized)} examples")
print(f"Validation dataset: {len(val_tokenized)} examples")

Map:   0%|          | 0/7999 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Train dataset: 7999 examples
Validation dataset: 1001 examples


In [10]:
# Check input truncation for BART classification
def check_truncation(data, max_input):
    """Check how many examples exceed the input token limit."""

    input_truncated = 0

    for instance in data:
        text = instance["dialects"]["natural"]

        tokens = tokenizer(
            text,
            return_length=True,
            padding=False,
            truncation=False,
        )["length"]

        if isinstance(tokens, list):
            tokens = tokens[0]

        if tokens > max_input:
            input_truncated += 1

    return input_truncated


input_trunc = check_truncation(
    raw_data,
    CONFIG["max_input_length"]
)

print(
    f"Input truncation: {input_trunc}/{len(raw_data)} "
    f"({input_trunc/len(raw_data)*100:.2f}%)"
)

TypeError: 'int' object is not subscriptable

In [ ]:
# Setup training arguments
training_args = TrainingArguments(
    output_dir="./bart_pde_results",
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    eval_strategy="epoch",
    save_strategy="epoch",
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=use_cuda,
)


# Data collator
# The Trainer will automatically place tensors on the selected device.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

print("Trainer initialized successfully")

Trainer initialized successfully


In [ ]:
# Train the model
print("Starting training...")
train_result = trainer.train()

print("\nTraining complete!")
print(f"Training loss: {train_result.training_loss:.4f}")

Starting training...


Epoch,Training Loss,Validation Loss
1,0.084263,0.000267
2,0.007829,0.000022
3,0.001347,0.000018


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight'].



Training complete!
Training loss: 0.0311


## Phase 3: Multitask BART Upgrade

This phase replaces the old family-only cross-dialect loop with the stronger multitask BART workflow.

What changes here:
- mixed-dialect training across natural, LaTeX, prefix, and postfix
- explicit dialect tags such as `[DIALECT_LATEX]`
- family classification plus operator prediction
- auxiliary structural targets: time order, spatial derivatives, nonlinearity, and spatial variable count


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import matplotlib.pyplot as plt

MULTITASK_OUTPUT_DIR = Path('outputs') / 'bart_multitask_pde' / 'instance' / 'mixed'
MULTITASK_METRICS_PATH = MULTITASK_OUTPUT_DIR / 'multitask_metrics.json'

print(f'Working directory: {Path.cwd()}')
print(f'Expected metrics path: {MULTITASK_METRICS_PATH}')

In [ ]:
REPORT_CMD = [
    sys.executable,
    'scripts/run_bart_multitask_pde.py',
    '--report-only',
    '--train-mode', 'mixed',
    '--split-mode', 'instance',
]

print(' '.join(REPORT_CMD))
subprocess.run(REPORT_CMD, check=True)

In [ ]:
TRAIN_CMD = [
    sys.executable,
    'scripts/run_bart_multitask_pde.py',
    '--train-mode', 'mixed',
    '--split-mode', 'instance',
]

# For a single-dialect comparison, switch to something like:
# TRAIN_CMD = [
#     sys.executable,
#     'scripts/run_bart_multitask_pde.py',
#     '--train-mode', 'single',
#     '--train-dialect', 'natural',
#     '--split-mode', 'instance',
# ]

print(' '.join(TRAIN_CMD))
subprocess.run(TRAIN_CMD, check=True)

In [ ]:
if not MULTITASK_METRICS_PATH.exists():
    raise FileNotFoundError(
        f'{MULTITASK_METRICS_PATH} does not exist yet. Run the training cell first.'
    )

with open(MULTITASK_METRICS_PATH) as f:
    multitask_summary = json.load(f)

multitask_val_df = pd.DataFrame(multitask_summary['val_metrics']).T.sort_index()
multitask_test_df = pd.DataFrame(multitask_summary['test_metrics']).T.sort_index()

ordered_cols = [
    'family_accuracy',
    'operator_micro_f1',
    'structure_accuracy',
    'time_order_accuracy',
    'first_spatial_accuracy',
    'second_spatial_accuracy',
    'nonlinear_accuracy',
    'spatial_var_accuracy',
]

multitask_val_df = multitask_val_df[ordered_cols]
multitask_test_df = multitask_test_df[ordered_cols]

In [ ]:
print('Validation metrics by evaluation dialect:')
display(multitask_val_df.style.format('{:.2%}'))

print('Test metrics by evaluation dialect:')
display(multitask_test_df.style.format('{:.2%}'))

In [ ]:
summary_rows = []
for dialect, row in multitask_test_df.iterrows():
    summary_rows.append({
        'Dialect': dialect,
        'Family Acc': row['family_accuracy'],
        'Operator F1': row['operator_micro_f1'],
        'Structure Acc': row['structure_accuracy'],
    })

summary_df = pd.DataFrame(summary_rows).set_index('Dialect').sort_values(
    by=['Family Acc', 'Operator F1', 'Structure Acc'], ascending=False
)

print('High-level multitask summary:')
display(summary_df.style.format('{:.2%}'))

best_family = multitask_test_df['family_accuracy'].idxmax()
best_operator = multitask_test_df['operator_micro_f1'].idxmax()
best_structure = multitask_test_df['structure_accuracy'].idxmax()

print(f"Best family dialect on test set: {best_family} ({multitask_test_df.loc[best_family, 'family_accuracy']:.2%})")
print(f"Best operator dialect on test set: {best_operator} ({multitask_test_df.loc[best_operator, 'operator_micro_f1']:.2%})")
print(f"Best structure dialect on test set: {best_structure} ({multitask_test_df.loc[best_structure, 'structure_accuracy']:.2%})")

In [ ]:
plot_df = multitask_test_df[['family_accuracy', 'operator_micro_f1', 'structure_accuracy']].copy()
plot_df.columns = ['Family Accuracy', 'Operator F1', 'Structure Accuracy']

ax = plot_df.plot(kind='bar', figsize=(10, 5), ylim=(0, 1.05), rot=0)
ax.set_title('Multitask BART Test Metrics by Evaluation Dialect')
ax.set_ylabel('Score')
ax.set_xlabel('Dialect')
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3)
plt.tight_layout()
plt.show()

In [ ]:
config_summary = pd.DataFrame([
    {'Parameter': 'Model', 'Value': multitask_summary['config']['model_name']},
    {'Parameter': 'Train Mode', 'Value': multitask_summary['config']['train_mode']},
    {'Parameter': 'Train Dialect', 'Value': multitask_summary['config']['train_dialect']},
    {'Parameter': 'Eval Dialects', 'Value': ', '.join(multitask_summary['config']['eval_dialects'])},
    {'Parameter': 'Split Mode', 'Value': multitask_summary['config']['split_mode']},
    {'Parameter': 'Batch Size', 'Value': multitask_summary['config']['batch_size']},
    {'Parameter': 'Epochs', 'Value': multitask_summary['config']['num_epochs']},
    {'Parameter': 'Learning Rate', 'Value': multitask_summary['config']['learning_rate']},
    {'Parameter': 'Max Input Length', 'Value': multitask_summary['config']['max_input_length']},
    {'Parameter': 'Operator Loss Weight', 'Value': multitask_summary['config']['operator_loss_weight']},
    {'Parameter': 'Structure Loss Weight', 'Value': multitask_summary['config']['structure_loss_weight']},
])

print('Run configuration:')
display(config_summary)

## Conclusion

This notebook now keeps the original single-dialect baseline training cells, but the main evaluation workflow is the multitask BART upgrade.

That makes the notebook better aligned with the actual research question: whether the model can learn PDE structure instead of only memorizing one dialect's surface form.